# Totimorphic lattice

This notebook constructs a totimorphic lattice by assembling cells from the [totimorphic cell notebook](./totimotphic_cell.ipynb).

In [1]:
# Project and symbolic tools
import heyoka as hy
import quivira as qv

# Numerics and visualization
import numpy as np
from _animation import animate_bar_model

# Building the lattice

The lattice is constructed, `n_x` and `n_y` are the number of cells in each direction of the grid.

In [2]:
model = qv.bar_joint_model()

# Number of cells in each direction
n_x = 4
n_y = 4

# Cell spacing
dx = 8
dy = 8

# Original cell geometry
base_point_pairs = [
    ((0, 0), (1, 4)),
    ((1, 4), (0, 8)),
    ((0, 8), (4, 7)),
    ((4, 7), (8, 8)),
    ((8, 8), (9, 4)),
    ((9, 4), (8, 0)),
    ((8, 0), (4, -1)),
    ((4, -1), (0, 0)),
]

# Keep track of already existing bars
existing_bars = set()

top_right_bar_index = None

for i in range(n_x):
    for j in range(n_y):

        offset_x = i * dx
        offset_y = j * dy

        for k, (start, end) in enumerate(base_point_pairs):

            new_start = (start[0] + offset_x, start[1] + offset_y)
            new_end = (end[0] + offset_x, end[1] + offset_y)

            bar_key = tuple(sorted((new_start, new_end)))

            if bar_key in existing_bars:
                continue

            # Identify bar 3 in the upper-right cell
            if i == n_x - 1 and j == n_y - 1 and k == 3:
                top_right_bar_index = len(model.bars)

            model.add_bar_from_point_pair(new_start, new_end)
            existing_bars.add(bar_key)

model.infer_joints()

model.add_support(bar_index=0, s=-0.5, dof_in_x=False, dof_in_y=False)

model.add_external_force(bar_index=top_right_bar_index, s=0.5, fx=hy.par[0], fy=hy.par[1])

Build the ODE system

In [3]:
ode_system, W = model.build_lagrange_eom(return_W=True)

hy.lagrangian took 0.215833 seconds
build_mass_matrix took 0.074162 seconds
build_constraint_jacobian took 0.049737 seconds


build_Jdqd took 0.134218 seconds
factorize M took 0.016825 seconds
force contribution took 0.000364 seconds
solve M @ G = J.T took 0.018201 seconds
my_matmul: A(192, 240) @ B(240, 192) -> C(192, 192)
build W took 0.015480 seconds
build rhs took 0.030257 seconds
gaussian_elimination took 0.082953 seconds
my_matmul: A(240, 192) @ B(192, 1) -> C(240, 1)
acceleration correction took 0.006273 seconds
find_constrained_accelerations took 0.171146 seconds


## Taylor integrator

The x and y components of the force are set as heyoka parameters

In [4]:
fx = 150.0
fy = 50.0

ta = hy.taylor_adaptive(ode_system, pars=[fx, fy], compact_mode=True)

## Simulation

In [5]:
state0 = model.get_initial_state()
model.check_constraint_fulfillment(state0)

In [6]:
ta.time = 0.0
ta.state[:] = state0
ta.pars[:] = [300.0, 100.0]

t_end = 4
tgrid = np.linspace(0.0, t_end, 401)
solution = ta.propagate_grid(tgrid)[-1]

Animate

In [7]:
# Use your existing 2D trajectory variable here:
animate_bar_model(model, solution, force_values=(fx, fy), save_animation=False)